# Notebook 00 — Introduction to Geoprivacy

## Pre-Cryptographic Approaches to Location Privacy

GPS coordinates are quasi-identifying even without names attached. A sequence of
locations — home, workplace, clinic, place of worship — can uniquely fingerprint
an individual. This notebook introduces the geoprivacy problem and two
pre-cryptographic approaches that represent the state of practice before
full cryptographic pipelines:

- **Donut geomasking** (NB00a): replace each location with a random point at
  a controlled distance from the original
- **H3 hex-grid binning** (NB00c): snap each location to the centroid of a
  hexagonal cell at a chosen resolution

Both approaches reduce location precision but make no cryptographic guarantees.
NB01-NB17 build the case for why a full cryptographic pipeline — pseudorandom
tile permutation, AEAD residual encryption, and display jitter — is needed when
stronger privacy guarantees are required.

**Three-part structure:**

- **Part 1** — The geoprivacy problem: quasi-identification from location data
- **Part 2** — Donut geomasking: random displacement within a spatial band
- **Part 3** — H3 hex-grid binning: aggregation into equal-area hexagonal cells


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium

from geoprivacy.donut_geomask import donut_geomask, distance_between_points
from geoprivacy.data_geomask import data_geomask
from geoprivacy.hexgrid import get_h3_address, visualize_hexagons


---
## 00.1  The Geoprivacy Problem

Geographic records are uniquely sensitive: even coarse location data can identify
individuals when combined with auxiliary information. The 1854 Soho cholera dataset
used throughout these notebooks is a canonical example — 250 death locations in a
small neighbourhood where building-level precision is sufficient to re-identify
residents.

Three properties make GPS coordinates especially risky as quasi-identifiers:

1. **Precision**: consumer GPS is accurate to 3-10 m, sufficient to identify a
   specific address or room within a building.
2. **Aggregability**: a time-series of locations (home at night, work by day,
   clinic on Tuesday) is far more identifying than any single fix.
3. **Linkability**: location records join easily to other datasets — cadastral
   maps, voter rolls, social media check-ins — without a shared key.

The two notebooks in this series (00a, 00c) demonstrate what happens when
locations are protected by simpler, non-cryptographic means.


In [2]:
df = pd.read_csv('data/cholera_deaths.csv')
pumps = pd.read_csv('data/pumps.csv')
print(f'Cholera deaths: {len(df)} locations, {df.DEATHS.sum()} total deaths')
print(f'Pumps: {len(pumps)} locations')
df.head(3)


Cholera deaths: 250 locations, 489 total deaths
Pumps: 8 locations


,FID,DEATHS,LON,LAT
0,0,3,-0.137930,51.513418
1,1,2,-0.137883,51.513361
2,2,1,-0.137853,51.513317


---
## 00.2  Original Locations — Soho 1854

The 250 death locations cluster tightly around the Broadwick Street pump.
At 1:3000 scale, individual building resolution is visible. Publishing these
coordinates without protection would directly expose home addresses.


In [3]:
m = folium.Map(location=[51.5134, -0.1365], zoom_start=16, tiles='cartodbpositron')
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row.LAT, row.LON],
        radius=max(2, row.DEATHS),
        color='#1f77b4', fill=True, fill_opacity=0.6,
        tooltip=f'Deaths: {row.DEATHS}'
    ).add_to(m)
for _, row in pumps.iterrows():
    folium.Marker(
        location=[row.LAT, row.LON],
        icon=folium.Icon(color='red', icon='tint', prefix='fa'),
        tooltip=row.Street
    ).add_to(m)
m


**Figure 00a** — Original 250 cholera death locations (blue circles, radius proportional
to deaths) and 8 water pump locations (red markers) in Soho, 1854. The tight cluster
around the Broadwick Street pump is clearly visible at this zoom level.


---
## 00.3  Donut Geomasking — Random Displacement

Donut geomasking replaces each true location with a point randomly chosen at a
distance between a minimum and maximum radius (the inner and outer donut radii).
The direction is also random. This ensures no geomasked point falls exactly at
the true location (inner radius > 0) while limiting how far the point can move
(outer radius sets the maximum displacement).

The `geoprivacy.donut_geomask` module implements this approach. Key parameters:

- **band_range**: `((min_lo, max_lo), (min_hi, max_hi))` — inner and outer radius
  ranges in metres. For fixed radii (e.g., 50-125 m), set both lo values equal
  and both hi values equal: `((50, 51), (125, 126))`.
- The function returns masked latitude, longitude, bearing, and distance.

NB00a demonstrates this in full on all 250 cholera deaths, with a re-identification
experiment showing ~1.6% of masked records can be recovered within 20 m.


In [4]:
BAND = ((50, 51), (125, 126))
pump_lat, pump_lon = 51.513341, -0.136668  # Broadwick Street pump

result = donut_geomask(band_range=BAND, orig_point=(pump_lat, pump_lon))
masked_lat = result['latitude']
masked_lon = result['longitude']
dist_m = result['distance'] * 1000

d = distance_between_points(
    orig=(pump_lat, pump_lon),
    dest=(masked_lat, masked_lon)
)
print(f'Original:  ({pump_lat:.6f}, {pump_lon:.6f})')
print(f'Geomasked: ({masked_lat:.6f}, {masked_lon:.6f})')
print(f'Displacement: {d["geodesic_m"]:.1f} m (bearing {result["bearing"]:d} deg)')


Original:  (51.513341, -0.136668)
Geomasked: (51.512744, -0.137500)
Displacement: 88.0 m (bearing 221 deg)


**Result** — The masked point is displaced 50-125 m from the true location in a random
direction. A re-identification attacker using the donut centroid as a prior can narrow
the true location to a ring-shaped search region. See NB00b for the re-identification
rate analysis.


---
## 00.4  H3 Hex-Grid Binning — Aggregation into Hexagonal Cells

H3 hex-grid binning snaps each true location to the centroid of an H3 hexagonal
cell at a chosen resolution. Multiple locations that fall within the same cell
share an identical masked coordinate — the cell centroid. This provides a form of
spatial k-anonymity: any record in a cell with k other records cannot be spatially
distinguished from those k records.

H3 hexagonal cells have more globally regular areas at each resolution than
Web Mercator square bins. At resolution 9, the average edge length is ~174 m
and the average cell area is ~0.105 km^2.

The trade-off is significant utility loss: all within-cell precision is gone
and the masked location is always the cell centroid, not a random displacement.

NB00c demonstrates H3 binning in full with multi-resolution analysis (resolutions
7-9) and Folium cell-boundary maps.


In [5]:
BROADWICK = (51.513341, -0.136668)

for res in [7, 8, 9]:
    cell = get_h3_address(BROADWICK, resolution=res)
    import h3 as _h3
    clat, clon = _h3.cell_to_latlng(cell)
    area_km2 = _h3.cell_area(cell, unit='km^2')
    print(f'Resolution {res}: cell={cell}  centroid=({clat:.4f},{clon:.4f})  area={area_km2:.4f} km^2')


Resolution 7: cell=87195da49ffffff  centroid=(51.5097,-0.1425)  area=4.6155 km^2
Resolution 8: cell=88195da499fffff  centroid=(51.5154,-0.1346)  area=0.6592 km^2
Resolution 9: cell=89195da498fffff  centroid=(51.5127,-0.1362)  area=0.0942 km^2


**Result** — At resolution 9, the Broadwick Street pump location maps to a cell whose
centroid is approximately 150 m from the true location. All deaths within the same
cell would share this centroid, providing spatial k-anonymity within the cell.


---
## 00.5  Limitations and the Case for Cryptographic Protection

Both approaches provide useful but bounded privacy guarantees:

| Approach | Re-ID risk | Utility loss | Key required | Tamper detectable |
|----------|-----------|-------------|--------------|-------------------|
| Donut geomasking | ~1.6% within 20 m | Low-medium | No | No |
| H3 hex-grid binning | ~0% within cell | High (centroid only) | No | No |
| Full PRP+AEAD pipeline (NB01-NB17) | ~0% | Minimal (jitter only) | Yes | Yes (AEAD) |

Three limitations shared by both baseline approaches:

1. **No cryptographic security**: an adversary who obtains the masking parameters
   can potentially reverse the mask or narrow the search region substantially.
2. **No tamper detection**: a geomasked record can be silently altered without
   any detectable integrity violation.
3. **No key-based access control**: anyone who has the masked dataset can attempt
   re-identification; there is no privileged decode tier.

The full pipeline in NB01-NB17 addresses all three limitations by adding
pseudorandom tile permutation (PRP), authenticated encryption (AEAD), and
a three-tier key privilege model.

**Reading path:**
- NB00a: donut geomasking in full on 250 cholera deaths
- NB00b: donut geomasking evaluation — re-ID rate, weighted mean center, performance sweep
- NB00c: H3 hex-grid binning at resolutions 7-9
- NB01: introduction to the four-step cryptographic pipeline that supersedes these approaches
